# 01 — Build / verify taxonomies

The paper uses three label taxonomies built from the 45+ RFMiD disease columns:

| Taxonomy | #labels | How it is built |
|---|---|---|
| **ODIR-7** | 7 (N,D,G,C,A,M,O) | Fixed clinical map (ODIR-style, no H) |
| **Cluster-C** | 10 | Supervised ResNet-50 column encoder → PCA-32 → k-means (k=10) |
| **Sup-K12** | 14 | Latent regrouping (k=12) + `otros` + `sano` |

Shipped CSVs under `data/taxonomies/` are the **source of truth** for the paper numbers.
This notebook verifies them (and, if `RETINA_DATA_ROOT` is set, can rebuild ODIR-7).
Full Cluster-C / Sup-K12 *retraining* of the encoder needs GPU + the full image set; the
shipped `centroids.npy` + maps already reproduce the paper tables.


In [ ]:
# --- cell 1: works on Colab and locally ---
from pathlib import Path
import os, sys, subprocess

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    # clone (or pull) the public repo; replace USER/REPO after you publish
    REPO_URL = os.environ.get("RETINA_REPO_URL", "https://github.com/TODO_USER/retina_multilabel.git")
    ROOT = Path("/content/retina_multilabel")
    if not (ROOT / "pyproject.toml").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])
    else:
        subprocess.call(["git", "-C", str(ROOT), "pull", "--ff-only"])
    os.chdir(ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
else:
    # local: walk up until we find the repo root
    HERE = Path.cwd().resolve()
    ROOT = HERE
    for cand in [HERE, *HERE.parents]:
        if (cand / "pyproject.toml").exists() and (cand / "scripts" / "01_build_taxonomies.py").exists():
            ROOT = cand
            break
    os.chdir(ROOT)
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))

print("IN_COLAB", IN_COLAB, "ROOT", ROOT)

In [ ]:
# --- verify shipped taxonomies (calls scripts/01_build_taxonomies.py) ---
import subprocess, sys
from pathlib import Path
rc = subprocess.call([sys.executable, str(ROOT / "scripts" / "01_build_taxonomies.py")], cwd=ROOT)
assert rc == 0, "taxonomy check failed"

In [ ]:
# --- show column → label maps + per-split counts ---
import json
from pathlib import Path
import pandas as pd

TAX = ROOT / "data" / "taxonomies"

def show_taxonomy(name: str, folder: str) -> None:
    root = TAX / folder
    cmap = json.loads((root / "cluster_map.json").read_text(encoding="utf-8"))
    labels = cmap.get("cluster_names_ordered") or [
        cmap["cluster_id_to_name"][str(i)] for i in range(cmap.get("k", 0))
    ]
    print(f"\n===== {name}  ({len(labels)} labels) =====")
    print("labels:", labels)
    col2 = cmap.get("column_to_cluster_id") or {}
    # pretty: column -> label name
    id2name = cmap.get("cluster_id_to_name") or {str(i): n for i, n in enumerate(labels)}
    rows = []
    for col, cid in sorted(col2.items(), key=lambda kv: (kv[1], kv[0])):
        rows.append({"rfmid_column": col, "cluster_id": cid, "label": id2name.get(str(cid), id2name.get(cid, cid))})
    map_df = pd.DataFrame(rows)
    try:
        from IPython.display import display
        display(map_df)
    except Exception:
        print(map_df.to_string(index=False))

    # per-split positive counts
    counts = []
    for split in ("train", "val", "test"):
        csv = root / "labels" / f"clustered_{split}.csv"
        if not csv.exists():
            # clustering_C_supervised uses same layout
            continue
        df = pd.read_csv(csv)
        labs = [c for c in df.columns if c not in ("ID", "Disease_Risk")]
        for lab in labs:
            counts.append({"split": split, "label": lab, "positives": int(df[lab].sum()), "n": len(df)})
    if counts:
        cdf = pd.DataFrame(counts)
        piv = cdf.pivot(index="label", columns="split", values="positives")
        print("positives per split:")
        try:
            from IPython.display import display
            display(piv)
        except Exception:
            print(piv.to_string())

# prefer cluster_c (dual_tax materialization) for the 10-label paper taxonomy
show_taxonomy("ODIR-7", "odir7")
show_taxonomy("Cluster-C", "cluster_c")
show_taxonomy("Sup-K12", "supervised_k12_sano")

In [ ]:
# --- self-check: shipped CSVs are readable and internally consistent ---
from pathlib import Path
import json, hashlib
import pandas as pd

TAX = ROOT / "data" / "taxonomies"
expected = {
    "odir7": 7,
    "cluster_c": 10,
    "supervised_k12_sano": 14,
}
all_ok = True
for name, k in expected.items():
    root = TAX / name
    cmap = json.loads((root / "cluster_map.json").read_text(encoding="utf-8"))
    labels = cmap["cluster_names_ordered"]
    assert len(labels) == k, (name, len(labels), k)
    for split, n_expect in [("train", 1920), ("val", 640), ("test", 640)]:
        df = pd.read_csv(root / "labels" / f"clustered_{split}.csv")
        assert len(df) == n_expect, (name, split, len(df))
        assert list(df.columns[:1]) == ["ID"]
        for lab in labels:
            assert lab in df.columns, (name, lab)
            assert set(df[lab].unique()) <= {0, 1}
    print(f"OK {name}: k={k}, splits 1920/640/640")

# fingerprint reference odir7 test labels (stable across clones)
odir7_test = TAX / "odir7" / "labels" / "clustered_test.csv"
h = hashlib.md5(odir7_test.read_bytes()).hexdigest()
print("odir7 test CSV md5:", h)
print("All shipped taxonomies match the expected paper layout.")

In [ ]:
# --- optional: rebuild ODIR-7 from RFMiD CSVs (needs RETINA_DATA_ROOT) ---
import os
from pathlib import Path

data_root = os.environ.get("RETINA_DATA_ROOT")
if not data_root:
    print("RETINA_DATA_ROOT not set — skip rebuild (shipped CSVs already match the paper).")
else:
    print("RETINA_DATA_ROOT =", data_root)
    print("To rebuild ODIR-7 exactly, use paper_train.odir_taxonomy helpers or the")
    print("archived script in retina_no_github/.../00_build_datasets.py --rebuild-odir7.")
    print("Cluster-C / Sup-K12 encoder training is GPU-heavy; keep the shipped maps for reproduction.")